# EmbedLab — Module 1 Training (Colab GPU runtime)

This notebook runs the **actual full training run** for the supervised contrastive encoder (Siamese + NT-Xent on STS-B). It exists because the local dev machine has no CUDA GPU, so smoke-testing happens locally (see `training/train_contrastive.py`) but real training happens here.

**Before running:** Runtime > Change runtime type > set Hardware accelerator to **GPU** (a free T4 is enough).

This notebook is **self-contained** — it duplicates the logic from `models/encoder.py`, `models/siamese.py`, `losses/nt_xent.py`, `data/prepare.py`, and `training/train_contrastive.py` inline, since the repo isn't pushed to a Git remote yet. Once it is, replace the class/function cells below with a `!git clone` + imports instead of keeping two copies in sync.

**After running:** download the checkpoint at the bottom and drop it into your local `checkpoints/contrastive_encoder.pt`, then run `evaluation/sts_eval.py` locally (cheap — forward passes only) to get the Spearman ρ for the results table.

In [ ]:
!pip install -q transformers datasets

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > GPU"
print("device:", torch.cuda.get_device_name(0))

In [ ]:
import time
from pathlib import Path

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from datasets import load_dataset

## Config

Mirrors `configs/default.yaml` — edit here if you want to try different hyperparameters for this run.

In [ ]:
config = {
    "model": {"backbone": "bert-base-uncased", "pooling": "mean", "max_length": 128},
    "data": {
        "sts_b_dataset": "mteb/stsbenchmark-sts",
        "cache_dir": "/content/data_cache",
        "positive_threshold": 3.5,
        "negative_threshold": 2.0,
    },
    "training": {
        "batch_size": 32,
        "num_epochs": 3,
        "learning_rate": 2.0e-5,
        "warmup_ratio": 0.1,
        "weight_decay": 0.01,
        "temperature": 0.05,
        "seed": 42,
    },
    "output": {"checkpoint_dir": "/content/checkpoints"},
}

## Model: TextEncoder + SiameseEncoder

(duplicated from `models/encoder.py` / `models/siamese.py`)

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, model_name="bert-base-uncased", pooling="mean"):
        super().__init__()
        if pooling not in ("mean", "cls"):
            raise ValueError(f"Unsupported pooling: {pooling!r}")
        self.pooling = pooling
        self.backbone = AutoModel.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.hidden_size = self.backbone.config.hidden_size

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state
        if self.pooling == "cls":
            return token_embeddings[:, 0]
        return self._mean_pool(token_embeddings, attention_mask)

    @staticmethod
    def _mean_pool(token_embeddings, attention_mask):
        mask = attention_mask.unsqueeze(-1).to(token_embeddings.dtype)
        summed = (token_embeddings * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts


class SiameseEncoder(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, input_ids_1, attention_mask_1, input_ids_2, attention_mask_2):
        return self.encoder(input_ids_1, attention_mask_1), self.encoder(input_ids_2, attention_mask_2)

    def encode_pairs(self, sentences_1, sentences_2, max_length=128, device="cpu"):
        tokenizer = self.encoder.tokenizer
        batch_1 = tokenizer(sentences_1, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        batch_2 = tokenizer(sentences_2, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        return self.forward(batch_1["input_ids"], batch_1["attention_mask"], batch_2["input_ids"], batch_2["attention_mask"])

## Loss: NT-Xent

(duplicated from `losses/nt_xent.py`)

In [ ]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_anchor, z_positive):
        z_anchor = F.normalize(z_anchor, dim=-1)
        z_positive = F.normalize(z_positive, dim=-1)
        sim_matrix = z_anchor @ z_positive.T / self.temperature
        labels = torch.arange(sim_matrix.size(0), device=sim_matrix.device)
        loss_a2p = F.cross_entropy(sim_matrix, labels)
        loss_p2a = F.cross_entropy(sim_matrix.T, labels)
        return (loss_a2p + loss_p2a) / 2

## Data: STS-B

(duplicated from `data/prepare.py`)

In [ ]:
def prepare_sts_b(dataset_id, cache_dir, positive_threshold=3.5, negative_threshold=2.0):
    raw = load_dataset(dataset_id, cache_dir=str(cache_dir))
    splits = {}
    for split_name in ("train", "validation", "test"):
        if split_name not in raw:
            continue
        df = raw[split_name].to_pandas()[["sentence1", "sentence2", "score"]].copy()
        df["sentence1"] = df["sentence1"].str.strip()
        df["sentence2"] = df["sentence2"].str.strip()
        df["score"] = df["score"].astype(float)
        df["label"] = -1
        df.loc[df["score"] >= positive_threshold, "label"] = 1
        df.loc[df["score"] <= negative_threshold, "label"] = 0
        splits[split_name] = df.reset_index(drop=True)
    return splits


class PositivePairDataset(Dataset):
    def __init__(self, df):
        self.sentences_1 = df.loc[df["label"] == 1, "sentence1"].tolist()
        self.sentences_2 = df.loc[df["label"] == 1, "sentence2"].tolist()

    def __len__(self):
        return len(self.sentences_1)

    def __getitem__(self, idx):
        return self.sentences_1[idx], self.sentences_2[idx]

## Training loop

(duplicated from `training/train_contrastive.py`)

In [ ]:
def build_optimizer(model, learning_rate, weight_decay):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        (no_decay if "bias" in name or "LayerNorm" in name else decay).append(param)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}],
        lr=learning_rate,
    )


def train(config):
    torch.manual_seed(config["training"]["seed"])
    device = torch.device("cuda")

    data_cfg = config["data"]
    splits = prepare_sts_b(
        data_cfg["sts_b_dataset"], data_cfg["cache_dir"],
        positive_threshold=data_cfg["positive_threshold"], negative_threshold=data_cfg["negative_threshold"],
    )
    train_dataset = PositivePairDataset(splits["train"])
    print(f"{len(train_dataset)} positive training pairs")

    train_cfg = config["training"]
    dataloader = DataLoader(train_dataset, batch_size=train_cfg["batch_size"], shuffle=True)

    encoder = TextEncoder(model_name=config["model"]["backbone"], pooling=config["model"]["pooling"]).to(device)
    siamese = SiameseEncoder(encoder)
    loss_fn = NTXentLoss(temperature=train_cfg["temperature"])

    optimizer = build_optimizer(encoder, train_cfg["learning_rate"], train_cfg["weight_decay"])
    total_steps = len(dataloader) * train_cfg["num_epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * train_cfg["warmup_ratio"]), num_training_steps=total_steps,
    )

    encoder.train()
    step = 0
    start = time.time()
    for epoch in range(train_cfg["num_epochs"]):
        for sentences_1, sentences_2 in dataloader:
            optimizer.zero_grad()
            emb_1, emb_2 = siamese.encode_pairs(list(sentences_1), list(sentences_2), max_length=config["model"]["max_length"], device=device)
            loss = loss_fn(emb_1, emb_2)
            loss.backward()
            optimizer.step()
            scheduler.step()
            step += 1
            if step % 10 == 0 or step == 1:
                print(f"epoch {epoch} step {step}/{total_steps} loss={loss.item():.4f} ({time.time() - start:.1f}s elapsed)")

    checkpoint_dir = Path(config["output"]["checkpoint_dir"])
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / "contrastive_encoder.pt"
    torch.save(encoder.state_dict(), checkpoint_path)
    print(f"saved checkpoint to {checkpoint_path}")
    return checkpoint_path

In [ ]:
checkpoint_path = train(config)

## Download the checkpoint

Save this file as `checkpoints/contrastive_encoder.pt` in your local repo, then run `uv run python -m evaluation.sts_eval` locally.

In [ ]:
from google.colab import files
files.download(str(checkpoint_path))